# HumanAI Detect - Claude (3. Uretici) Takviye Ornekleri: On Isleme (Asama 2, Colab GPU)

**Amac:** Claude Sonnet 5 ile uretilen `ai_raw_anthropic` (209 ornek -- 183 API + 26 elle) ve
back-translation ile uretilen `ai_humanized_anthropic` (209 ornek) havuzlarini Stanza
dilbilimsel analiz + BERTurk/DistilBERT capraz-perplexity + GPT2-Turkish token-rank +
burstiness ile isler (bkz. proje notlari, 2026-07-19/20 -- Claude-yazimi metin ilk testte
%64 oraninda yanlis 'human' cikmisti, cozum icin Claude 3. uretici olarak eklendi).

**Neden Colab:** Yerel CPU'da tek ornek icin BERT-tabanli pseudo-perplexity ~200 saniye
surer -- burada uc ayri transformer modeli (2 perplexity + 1 causal LM token-rank) calistigi
icin CPU'da daha da yavas olur. GPU'da tahmini ~15-20 sn/ornek, 418 ornek (iki havuz) icin
toplam **~1.7-2.3 saat** (GPT-4o-mini'nin 2000 orneklik ~9-11 saatinden cok daha kisa,
Claude verisi hacim olarak kucuk).

**ONEMLI -- once kontrol et:** Bu notebook GitHub'daki kodu ceker. Yerelde YENI kod
degisikligi yaparsan (preprocess.py TOPUP_LABELS, data_sources.yaml, llm_generators.py)
Colab'da calistirmadan once mutlaka commit+push et.

**Ana veriye DOKUNMAZ:** Bu notebook `data/interim/ai_raw_anthropic/` ve
`data/interim/ai_humanized_anthropic/` altina AYRI dosyalar uretir -- mevcut
`data/interim/{ai_raw,ai_humanized}/` (Qwen+GPT-4o-mini ana havuzu) degismez. Birlestirme
islemi sonraki adimda (build_features'tan once) yerelde yapilacak.

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> GPU (T4 yeterli) secip tekrar dene.')

In [ ]:
# 2. Repo klonla (guncel kod)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect
!git log --oneline -3

In [ ]:
# 3. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q

In [ ]:
# 4. Stanza Turkce modelini indir (pip install'dan SONRA ayri hucrede)
import stanza
stanza.download('tr')

In [ ]:
# 5. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 6. Kaynak dosyalari yerel makineden yukle: ai_raw_anthropic.jsonl VE ai_humanized_anthropic.jsonl
# (ikisini birden secebilirsin, acilan pencerede coklu secim yap)
from pathlib import Path
from google.colab import files

Path('data/raw/ai_raw_anthropic').mkdir(parents=True, exist_ok=True)
Path('data/raw/ai_humanized_anthropic').mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if 'ai_humanized_anthropic' in name:
        Path(name).rename('data/raw/ai_humanized_anthropic/ai_humanized_anthropic.jsonl')
    elif 'ai_raw_anthropic' in name:
        Path(name).rename('data/raw/ai_raw_anthropic/ai_raw_anthropic.jsonl')

for label in ['ai_raw_anthropic', 'ai_humanized_anthropic']:
    p = Path(f'data/raw/{label}/{label}.jsonl')
    if p.exists():
        n = sum(1 for _ in open(p, encoding='utf-8'))
        print(f'{label}: {n} ornek yuklendi')
    else:
        print(f'{label}: UYARI, dosya yok!')

In [ ]:
# 6b. (SADECE onceki bir Colab oturumundan kaldiysa) kismi interim checkpoint'lerini
# geri yukle -- ilk kez calistiriyorsan bu hucreyi ATLA/calistirma.
from pathlib import Path
from google.colab import files

Path('data/interim/ai_raw_anthropic').mkdir(parents=True, exist_ok=True)
Path('data/interim/ai_humanized_anthropic').mkdir(parents=True, exist_ok=True)

print("Onceki oturumdan kalan interim ai_raw_anthropic.jsonl / ai_humanized_anthropic.jsonl dosyalarini sec")
uploaded = files.upload()
for name in uploaded:
    if 'ai_humanized_anthropic' in name:
        Path(name).rename('data/interim/ai_humanized_anthropic/ai_humanized_anthropic.jsonl')
    elif 'ai_raw_anthropic' in name:
        Path(name).rename('data/interim/ai_raw_anthropic/ai_raw_anthropic.jsonl')
print('checkpoint(ler) yuklendi, kaldigi yerden devam edilecek.')

In [ ]:
# 7a. ai_raw_anthropic on isleme (209 ornek, checkpoint'li -- kesinti olursa ayni
# hucreyi tekrar calistirmak yeterli, tamamlananlar atlanir)
!python scripts/preprocess.py --label ai_raw_anthropic

In [ ]:
# 7b. Sonucu indir
from pathlib import Path
from google.colab import files

src = Path('data/interim/ai_raw_anthropic/ai_raw_anthropic.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek -> indiriliyor')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')

In [ ]:
# 8a. ai_humanized_anthropic on isleme (209 ornek, checkpoint'li)
!python scripts/preprocess.py --label ai_humanized_anthropic

In [ ]:
# 8b. Sonucu indir
from pathlib import Path
from google.colab import files

src = Path('data/interim/ai_humanized_anthropic/ai_humanized_anthropic.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek -> indiriliyor')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')

## Yerel Makineye Aktarim

Yukaridaki hucreler `ai_raw_anthropic.jsonl` ve `ai_humanized_anthropic.jsonl`'i tarayicinin
varsayilan indirme klasorune (Downloads) indirir.

Colab bittikten sonra (tamamen bitmese de, kismi sonuc da olsa), kendi makinende:
1. Indirilen `ai_raw_anthropic.jsonl`'i `data/interim/ai_raw_anthropic/ai_raw_anthropic.jsonl`'e koy.
2. Indirilen `ai_humanized_anthropic.jsonl`'i `data/interim/ai_humanized_anthropic/ai_humanized_anthropic.jsonl`'e koy.
3. Bana haber ver -- ana `data/interim/{ai_raw,ai_humanized}`'e birlestirip
   `build_features.py` + yeniden egitim adimlarina gecerim.

**Oturum koparsa:** Indirdigin kismi dosyayi yeni bir Colab oturumunda 6b. hucreyle geri
yukleyip ayni komutu (7a/8a) tekrar calistirarak kaldigi yerden devam edebilirsin --
sifirdan baslamana GEREK YOK (checkpoint id bazinda calisir).